In [17]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import numpy as np
import time
import pandas as pd


In [18]:
def extraer_datos_meteorologicos(fecha):
    url = f"https://x-y.es/aemet/est-3129-madrid-barajas?fecha={fecha}"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Error al acceder a la página: {response.status_code}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table")
    
    if not table:
        print(f"No se encontró una tabla de datos para la fecha {fecha}")
        return None
    
    rows = table.find_all("tr")[1:]
    data = []
    for row in rows:
        columns = row.find_all("td")
        values = [col.text.strip() if col.text.strip() != "-" else np.nan for col in columns]
        data.append(values)

    df = pd.DataFrame(data, columns=["Hora", "Precipitación", "Temperatura", "Humedad",
                                     "Viento", "Dirección", "Viento máximo",
                                     "Dirección viento máximo", "Temperatura mínima",
                                     "Temperatura máxima"])
    
    return df

In [19]:
import time  # Para agregar pausas entre solicitudes

def extraer_datos_rango(fecha_inicio, fecha_fin):
    start_date = datetime.strptime(fecha_inicio, "%Y-%m-%d")
    end_date = datetime.strptime(fecha_fin, "%Y-%m-%d")

    all_data = []
    current_date = start_date
    
    while current_date <= end_date:
        fecha_str = current_date.strftime("%Y-%m-%d")
        print(f"Extrayendo datos para {fecha_str}...")

        df_dia = extraer_datos_meteorologicos(fecha_str)
        if df_dia is not None:
            df_dia["Fecha"] = fecha_str
            all_data.append(df_dia)

        # Pausa para evitar bloqueos (ajusta el tiempo si sigue bloqueando)
        #time.sleep(5)  # Espera 5 segundos entre cada solicitud

        current_date += timedelta(days=1)

    if all_data:
        df_total = pd.concat(all_data, ignore_index=True)
        return df_total
    else:
        print("No se encontraron datos en el rango especificado.")
        return None


In [20]:
fecha_inicio = "2024-11-07"
fecha_fin = "2025-01-31"

df_total = extraer_datos_rango(fecha_inicio, fecha_fin)

# Mostrar los primeros datos si la extracción fue exitosa
if df_total is not None:
    print(df_total.head())

Extrayendo datos para 2024-11-07...
Extrayendo datos para 2024-11-08...
Extrayendo datos para 2024-11-09...
Extrayendo datos para 2024-11-10...
Extrayendo datos para 2024-11-11...
Extrayendo datos para 2024-11-12...
Extrayendo datos para 2024-11-13...
Extrayendo datos para 2024-11-14...
Extrayendo datos para 2024-11-15...
Extrayendo datos para 2024-11-16...
Extrayendo datos para 2024-11-17...
Extrayendo datos para 2024-11-18...
Extrayendo datos para 2024-11-19...
Extrayendo datos para 2024-11-20...
Extrayendo datos para 2024-11-21...
Extrayendo datos para 2024-11-22...
Extrayendo datos para 2024-11-23...
Extrayendo datos para 2024-11-24...
Extrayendo datos para 2024-11-25...
Extrayendo datos para 2024-11-26...
Extrayendo datos para 2024-11-27...
Extrayendo datos para 2024-11-28...
Extrayendo datos para 2024-11-29...
Extrayendo datos para 2024-11-30...
Extrayendo datos para 2024-12-01...
Extrayendo datos para 2024-12-02...
Extrayendo datos para 2024-12-03...
Extrayendo datos para 2024-1

In [22]:
df_total

,Hora,Precipitación,Temperatura,Humedad,Viento,Dirección,Viento máximo,Dirección viento máximo,Temperatura mínima,Temperatura máxima,Fecha
0,01:00,"0,00","11,00","92,00","1,50",Noroeste 320°,"3,10",Noroeste 310°,"11,00","11,60",2024-11-07
1,02:00,"0,00","10,30","93,00","2,00",Norte 340°,"2,60",Norte 340°,"10,30","11,00",2024-11-07
2,03:00,"0,00","10,00","93,00","1,90",Norte 350°,"3,10",Norte 340°,"10,00","10,40",2024-11-07
3,04:00,"0,00","9,60","93,00","1,40",Norte 340°,"3,10",Norte 350°,"9,60","10,10",2024-11-07
4,05:00,"0,00","8,90","96,00","1,50",Noroeste 310°,"2,10",Noroeste 320°,"8,80","9,50",2024-11-07
...,...,...,...,...,...,...,...,...,...,...,...
1202,20:00,"0,00","6,50","78,00","1,70",Este 70°,"2,60",Noreste 60°,"6,30","7,80",2024-12-27
1203,21:00,"0,00","5,70","81,00","2,10",Noroeste 330°,"3,10",Noroeste 330°,"5,30","6,30",2024-12-27
1204,22:00,"0,00","4,40","88,00","2,30",Norte 360°,"3,10",Noroeste 310°,"4,30","5,00",2024-12-27
1205,23:00,"0,00","3,60","90,00","2,70",Norte 340°,"3,60",Norte 350°,"3,40","4,30",2024-12-27


In [23]:
print(df_total["Fecha"].unique())

['2024-11-07' '2024-11-08' '2024-11-09' '2024-11-10' '2024-11-11'
 '2024-11-12' '2024-11-13' '2024-11-14' '2024-11-15' '2024-11-16'
 '2024-11-17' '2024-11-18' '2024-11-19' '2024-11-20' '2024-11-21'
 '2024-11-22' '2024-11-23' '2024-11-24' '2024-11-25' '2024-11-26'
 '2024-11-27' '2024-11-28' '2024-11-29' '2024-11-30' '2024-12-01'
 '2024-12-02' '2024-12-03' '2024-12-04' '2024-12-05' '2024-12-06'
 '2024-12-07' '2024-12-08' '2024-12-09' '2024-12-10' '2024-12-11'
 '2024-12-12' '2024-12-13' '2024-12-14' '2024-12-15' '2024-12-16'
 '2024-12-17' '2024-12-18' '2024-12-19' '2024-12-20' '2024-12-21'
 '2024-12-22' '2024-12-23' '2024-12-24' '2024-12-25' '2024-12-26'
 '2024-12-27']


del 28 incluido al 31 incluido no salen de diciembre

In [24]:
fecha_inicio = "2024-12-28"
fecha_fin = "2025-01-31"

df_2 = extraer_datos_rango(fecha_inicio, fecha_fin)

# Mostrar los primeros datos si la extracción fue exitosa
if df_total is not None:
    print(df_total.head())

Extrayendo datos para 2024-12-28...
Extrayendo datos para 2024-12-29...
Extrayendo datos para 2024-12-30...
Extrayendo datos para 2024-12-31...
Extrayendo datos para 2025-01-01...
Extrayendo datos para 2025-01-02...
Extrayendo datos para 2025-01-03...
Extrayendo datos para 2025-01-04...
Extrayendo datos para 2025-01-05...
Extrayendo datos para 2025-01-06...
Extrayendo datos para 2025-01-07...
Extrayendo datos para 2025-01-08...
Extrayendo datos para 2025-01-09...
Extrayendo datos para 2025-01-10...
Extrayendo datos para 2025-01-11...
Extrayendo datos para 2025-01-12...
Extrayendo datos para 2025-01-13...
Extrayendo datos para 2025-01-14...
Extrayendo datos para 2025-01-15...
Extrayendo datos para 2025-01-16...
Extrayendo datos para 2025-01-17...
Extrayendo datos para 2025-01-18...
Extrayendo datos para 2025-01-19...
Extrayendo datos para 2025-01-20...
Extrayendo datos para 2025-01-21...
Extrayendo datos para 2025-01-22...
Extrayendo datos para 2025-01-23...
Extrayendo datos para 2025-0

In [25]:
print(df_2["Fecha"].unique())

['2024-12-28' '2024-12-29' '2024-12-30' '2024-12-31' '2025-01-01'
 '2025-01-02' '2025-01-03' '2025-01-04' '2025-01-05' '2025-01-06'
 '2025-01-07' '2025-01-08' '2025-01-09' '2025-01-10' '2025-01-11'
 '2025-01-12' '2025-01-13' '2025-01-14' '2025-01-15' '2025-01-16'
 '2025-01-17' '2025-01-18' '2025-01-19' '2025-01-20' '2025-01-21'
 '2025-01-22' '2025-01-23' '2025-01-24' '2025-01-25' '2025-01-26'
 '2025-01-27' '2025-01-28' '2025-01-29' '2025-01-30' '2025-01-31']


In [26]:
# Unir los DataFrames
df_final = pd.concat([df_total, df_2], ignore_index=True)

# Eliminar duplicados si existen
df_final = df_final.drop_duplicates(subset=["Fecha", "Hora"], keep="first")

# Ordenar por fecha y hora
df_final = df_final.sort_values(by=["Fecha", "Hora"]).reset_index(drop=True)

# Mostrar los primeros valores
print(df_final.head())

    Hora Precipitación Temperatura Humedad Viento      Dirección  \
0  00:00          0,00       10,90   96,00   1,40     Norte 350°   
1  01:00          0,00       11,00   92,00   1,50  Noroeste 320°   
2  02:00          0,00       10,30   93,00   2,00     Norte 340°   
3  03:00          0,00       10,00   93,00   1,90     Norte 350°   
4  04:00          0,00        9,60   93,00   1,40     Norte 340°   

  Viento máximo Dirección viento máximo Temperatura mínima Temperatura máxima  \
0          2,10              Norte 340°              10,90              11,40   
1          3,10           Noroeste 310°              11,00              11,60   
2          2,60              Norte 340°              10,30              11,00   
3          3,10              Norte 340°              10,00              10,40   
4          3,10              Norte 350°               9,60              10,10   

        Fecha  
0  2024-11-07  
1  2024-11-07  
2  2024-11-07  
3  2024-11-07  
4  2024-11-07  


In [27]:
print(df_final["Fecha"].unique())

['2024-11-07' '2024-11-08' '2024-11-09' '2024-11-10' '2024-11-11'
 '2024-11-12' '2024-11-13' '2024-11-14' '2024-11-15' '2024-11-16'
 '2024-11-17' '2024-11-18' '2024-11-19' '2024-11-20' '2024-11-21'
 '2024-11-22' '2024-11-23' '2024-11-24' '2024-11-25' '2024-11-26'
 '2024-11-27' '2024-11-28' '2024-11-29' '2024-11-30' '2024-12-01'
 '2024-12-02' '2024-12-03' '2024-12-04' '2024-12-05' '2024-12-06'
 '2024-12-07' '2024-12-08' '2024-12-09' '2024-12-10' '2024-12-11'
 '2024-12-12' '2024-12-13' '2024-12-14' '2024-12-15' '2024-12-16'
 '2024-12-17' '2024-12-18' '2024-12-19' '2024-12-20' '2024-12-21'
 '2024-12-22' '2024-12-23' '2024-12-24' '2024-12-25' '2024-12-26'
 '2024-12-27' '2024-12-28' '2024-12-29' '2024-12-30' '2024-12-31'
 '2025-01-01' '2025-01-02' '2025-01-03' '2025-01-04' '2025-01-05'
 '2025-01-06' '2025-01-07' '2025-01-08' '2025-01-09' '2025-01-10'
 '2025-01-11' '2025-01-12' '2025-01-13' '2025-01-14' '2025-01-15'
 '2025-01-16' '2025-01-17' '2025-01-18' '2025-01-19' '2025-01-20'
 '2025-01-

In [28]:
df_final

,Hora,Precipitación,Temperatura,Humedad,Viento,Dirección,Viento máximo,Dirección viento máximo,Temperatura mínima,Temperatura máxima,Fecha
0,00:00,"0,00","10,90","96,00","1,40",Norte 350°,"2,10",Norte 340°,"10,90","11,40",2024-11-07
1,01:00,"0,00","11,00","92,00","1,50",Noroeste 320°,"3,10",Noroeste 310°,"11,00","11,60",2024-11-07
2,02:00,"0,00","10,30","93,00","2,00",Norte 340°,"2,60",Norte 340°,"10,30","11,00",2024-11-07
3,03:00,"0,00","10,00","93,00","1,90",Norte 350°,"3,10",Norte 340°,"10,00","10,40",2024-11-07
4,04:00,"0,00","9,60","93,00","1,40",Norte 340°,"3,10",Norte 350°,"9,60","10,10",2024-11-07
...,...,...,...,...,...,...,...,...,...,...,...
2042,19:00,"0,00","7,70","48,00","1,50",Sureste 130°,"2,60",Sureste 120°,"7,60","9,70",2025-01-31
2043,20:00,"0,00","5,50","60,00","2,20",Noreste 40°,"3,10",Norte 10°,"5,50","7,50",2025-01-31
2044,21:00,"0,00","4,70","60,00","0,00",NaN,"3,10",Este 70°,"4,60","6,00",2025-01-31
2045,22:00,"0,00","4,20","59,00","0,90",Norte 20°,"2,60",Norte 10°,"4,20","5,30",2025-01-31


In [30]:
ruta_guardado = "../datos_meteorologicos.csv"  
df_final.to_csv(ruta_guardado, index=False)